# Testbench for CRUD operations using atlassian-python-api package

<table>
    <tr>
        <td><img src="https://upload.wikimedia.org/wikipedia/commons/f/f8/Python_logo_and_wordmark.svg" width="25%" /></td>
        <td><img src="https://www.atlassian.com/de/dam/jcr:1b7f4009-27d0-4882-8629-cecae97fc00f/Confluence-blue.svg" width="25%" /> </td>
    </tr>
</table>

This is the sandbox to evaluate and experiment with the official Atlassian python API: **atlassian-python-api**.

The API is hosted on github: https://github.com/atlassian-api/atlassian-python-api

The API documentation can be found here: https://atlassian-python-api.readthedocs.io/confluence.html

Required packages in your python environment:
- oauthlib
- requests-oauthlib
- deprecated

Install dependencies with this command:
`pip install -r requirements.txt`

The API is embedded as a **git submodule** in the 'lib' folder next to this notebook.
Use `git submodule update --init` to fetch submodules after a checkout without `--recursive` option.

`git submodule add -f https://github.com/atlassian-api/atlassian-python-api.git notebooks/Sandbox/confluence-python-api/lib`

In [ ]:
module = 'https://github.com/atlassian-api/atlassian-python-api'
library = 'lib/atlassian-python-api'

# Load configuration and credentials
Be aware to not commit your credentials!

In [ ]:
import yaml
with open('private.yaml') as f:
    config = yaml.safe_load(f)

conf_conf = config['confluence']
assert conf_conf
assert len(conf_conf['apiurl']) > 0
space_key = conf_conf['space']
root_page = conf_conf['rootpage']

{'apiurl': conf_conf['apiurl'], 'space_key': space_key, 'root_page': root_page}

In [ ]:
confluence_username = conf_conf.get('username')
confluence_password = conf_conf.get('password')

# Dependencies
Add atlassian-python-api git subrepo located inside this project

In [ ]:
import os
import sys
from datetime import datetime

import oauthlib.oauth1
import requests_oauthlib
import deprecated

## Add Confluence API to python path

In [ ]:
sys.path.insert(0, os.path.abspath(library))
from atlassian import Confluence
sys.path

# Helper functions

In [ ]:
import logging
log = logging.getLogger(__name__)

from requests.exceptions import HTTPError
from colorama import Fore, Style

def print_http_error_details(e: HTTPError):
    print(Fore.RED + e.response.content.decode('utf-8'))
    from pprint import pprint
    print(Fore.YELLOW + str(vars(e)))
    pprint(vars(e.response))
    result = e.response.raw
    pprint(vars(result))
    print(Style.RESET_ALL)
    
def trans(node: dict, lang: str ='de'):
    # extract language
    return node[lang]

# Check access to confluence

In [ ]:
log.warning('Accessing confluence space {} using user: {}'.format(space_key, confluence_username))
confluence = Confluence(url=conf_conf['apiurl'], username=confluence_username, password=confluence_password, cloud=True)
try:
    root_page_id = confluence.get_page_id(space_key, root_page)

except HTTPError as e:
    #print_http_error_details(e)
    raise Exception(e.response.content.decode('utf-8'))
    
root_page_id

In [ ]:
exists = confluence.page_exists(space_key, root_page)
assert exists, 'Missing root page \'{}\' required to add / update content'.format(root_page)

## Map the page name to the page ID

In [ ]:
root_page_id = confluence.get_page_id(space_key, root_page)
root_page_id

## Smoketest: Create a new child page

In [ ]:
stamp_now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
smoketest_child_page_id = None
try:
    status = confluence.create_page(space=conf_conf['space'], 
                                    title='confluence-python-api test page {}'.format(stamp_now),
                                    parent_id=root_page_id,
                                    body=('This page is beeing created using the'
                                          '<a href="{}">confluence-python-api</a> package.'
                                          '<br/>Created {}'.format(module, stamp_now)))
    smoketest_child_page_id = status['id']
    confluence.set_page_label(smoketest_child_page_id, 'test-delete-freely')
    print(status)

except HTTPError as e:
    #print_http_error_details(e)
    raise Exception(e.response.content.decode('utf-8'))

In [ ]:
try:
    page_information = confluence.get_page_by_id(status['id'], expand='ancestors,version,history,labels,body.storage')
except HTTPError as e:
    #print_http_error_details(e)
    raise Exception(e.response.content.decode('utf-8'))

page_information

In [ ]:
page_information['ancestors'][-1]

## Smoketest: Delete page by ID

In [ ]:
confluence.remove_page(smoketest_child_page_id, recursive=True)
log.warning('Page {} and all of it''s children have been deleted'.format(smoketest_child_page_id))

## Fetch existing child pages of root 

In [ ]:
child_pages = confluence.get_page_child_by_type(root_page_id, 'page')
log.warning('Found {} child pages'.format(len(child_pages)))
list(map(lambda x: ( x['id'] ), child_pages))

# Load the information model

In [ ]:
import json

test_json_file = config['json']
with open(test_json_file, 'r') as f:
    log.info('Loading information model from {}'.format(test_json_file))
    im = json.load(f)

assert '"type": "logical"' in json.dumps(im), "Expecting model type declaration" 

## First 5 entities

In [ ]:
entities = im['entities']
for entity_key in list(entities)[:5]:
    entity = entities[entity_key]
    print('{}: {}'.format(entity_key, trans(entity['name'])))